# Sweeping a single angle

This notebook walks us through a single-angle solve of the transport operator in several geometries, showing how to construct domains 


In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu" 
import jax
import jax.numpy as jnp
import numpy as onp

We are first going to create a simple slab domain in 1D:

In [ ]:
n_vertices = 7

vertices     = jnp.linspace(0, 1, n_vertices)[:, None]
connectivity = jnp.array([[i, i + 1] for i in range(n_vertices - 1)])
mesh         = (vertices, connectivity)

from jax_sn.domain import CrossSectionData, CrossSectionsIndexed, Domain
from jax_sn.solution_domain import BasixLagrangianSimplex

cross_section = CrossSectionData(
    total     = jnp.array([[5.0]]), 
    scattering= jnp.array([[[[0.0]]]]))

cross_sections_indexed = CrossSectionsIndexed(indices = jnp.full((n_vertices - 1,), 0), cross_section_data = cross_section)

domain = Domain.from_mesh_and_cross_sections(mesh, BasixLagrangianSimplex(1,1).face_template, cross_sections_indexed) # face template is [[0],[1]] first face is first node, second is second.

Let's pick a particular angle to invert the transport operator

In [ ]:
angle = jnp.array([-1.0, 0.0, 0.0]) 

Technically, a transport operator may require different data: a level-based sweep is different then a topologically ordered one. 
Here just create the basic variant:


In [ ]:
from jax_sn.operators.transport_operator import BasicSweepData

sweep_data = BasicSweepData.from_domain_angle(domain, angle)

print("Data structure         ", sweep_data)
print("Levels of each element:", sweep_data.levels)
print("Topological sort:      ", sweep_data.levels_argsort)

Unsurprisingly, with a negative angle, the level of the left-most element is 18 (the highest) while the topological sort determines that 
the first element to be solved is 18 (i.e. the right-most-element).

We can also visualize this (later we will do the same for 2D which is actually interesting):

In [ ]:
from jax_sn.tools.plotly import plot_domain
plot_domain(domain, show_graph=True, graph_angle = jnp.array([1.0,0.0,0.0]), show_levels=True)

Before creating a transport operator, we will have to define what kind of finite element will be used and precompute all the spatial data and FEM matrices. This is done by creating a `SolutionDomain` object.

In [ ]:
from jax_sn.solution_domain import BasixLagrangianSimplex, SolutionDomain

element_order   = 1
element         = BasixLagrangianSimplex(element_order, 1)
solution_domain = SolutionDomain.from_element_and_domain(element, domain)
print(solution_domain)

Now let's actually create the transport operator. To know which kind of SweepData we require, we have to select a strategy after which we can construct it.

Here it's just the basic version!

In [ ]:
from jax_sn.operators.transport_operator import TransportOperatorSingleAngle, VmapForward, VmapSweep, TransportData

inversion_strategy = VmapSweep()
print("Required sweep data type:", inversion_strategy.default_sweep_data_type())
sweep_data = inversion_strategy.default_sweep_data_type().from_domain_angle(domain, angle)

transport_data     = TransportData(spatial_data = solution_domain.spatial_data, 
                                   energy_group = 0,
                                   angle        = angle,
                                   sweep_data   = sweep_data)

transport_operator = TransportOperatorSingleAngle(data = transport_data, 
                                                  strategy = VmapForward(),
                                                  inversion_strategy = VmapSweep())




This is somewhat cumbersome, so there are convenience functions to directly do this from just a solution domain and strategies:

In [ ]:
from jax_sn.operator_creation import create_transport_operator
transport_operator_2 = create_transport_operator(solution_domain, angle, VmapForward(), VmapSweep(), energy_group = 0)
print("Equal trees:", jax.tree.all(jax.tree.map(jnp.array_equal, transport_operator, transport_operator_2)))

## Inverting the transport operator

Inverting the transport operator is easy! 

A detail is that the transport operator inversion is not enough to obtain a solution; the source has to be first operated on by the mass operator. This is because the system is defined as 

$$ \mathbf{T} \psi = \mathbf{H} q $$

such that 

$$ \psi = \mathbf{T}^{-1} \mathbf{H} q $$

In [ ]:
rhs = jnp.ones_like(solution_domain.elements_basis_shape)

from jax_sn.operators.mass_operator import MassOperator, MassData, LaxMass

massop = MassOperator(MassData(solution_domain.spatial_data.fem_matrices, 
                               solution_domain.spatial_data.precomputed_geometric_data), 
                      strategy = LaxMass(), additional_shape = ())


psi = transport_operator.inv(massop.mv(rhs))

Let's plot it together with the analytical solution:

$$ \psi(x) = \begin{cases}\frac{q}{\sigma_t} \left( 1 - e^{-\sigma_t x} \right) \quad \text{for } \mu > 0\\ \frac{q}{\sigma_t} \left( 1 - e^{-\sigma_t (1-x)} \right) \quad \text{for } \mu < 0\end{cases} $$

In [ ]:
def analytical_solution(x, angle, sigma_t, q ):    
    return jnp.where(angle[0] >0, q/sigma_t * (1.0 - jnp.exp(-sigma_t * x)), q/sigma_t * (1.0 - jnp.exp(-sigma_t * (1.0 - x))))

In [ ]:
from jax_sn.tools.plotly import plot_1d_solution

plot_1d_solution(
    solution_domain,
    psi,
    reference=lambda x: analytical_solution(x, angle, cross_section.total[0, 0], 1.0),
    yaxis_title="ψ",
)

The discontinuituous elements can be seen clearly here! It also matches the general shape of the analytical solution. Let's use a higher order finite element now:

In [ ]:

solution_domain_p3    = SolutionDomain.from_element_and_domain(BasixLagrangianSimplex(3,1), domain)
transport_operator_p3 = create_transport_operator(solution_domain_p3, angle, VmapForward(), VmapSweep(), energy_group = 0)
mass_operator_p3      = MassOperator(MassData(solution_domain_p3.spatial_data.fem_matrices,
                                                solution_domain_p3.spatial_data.precomputed_geometric_data), 
                                        strategy = LaxMass(), additional_shape = ())

rhs_p3 = jnp.ones_like(solution_domain_p3.elements_basis_shape)

psi_p3 = transport_operator_p3.inv(mass_operator_p3.mv(rhs_p3))

plot_1d_solution(
    solution_domain_p3,
    psi_p3,
    reference=lambda x: analytical_solution(x, angle, cross_section.total[0, 0], 1.0),
    yaxis_title="ψ",
)


This matches a lot closer!

## 2D cross section

We will do the same problem for a circular cross section now of a Miller-like tokamak using the `jax_sbgeom` package.

In [ ]:
from jax_sbgeom.flux_surfaces import ParametrisedSurface
import jax_sbgeom as jsb

class MillerTokamak(ParametrisedSurface):

    rminor : jnp.ndarray
    rmajor : jnp.ndarray
    elongation : jnp.ndarray
    triangularity : jnp.ndarray

    def cylindrical_position(self, s, theta, phi):

        s,theta,phi = jnp.broadcast_arrays(s,theta,phi)
        
        return jnp.stack([
            self.rmajor + self.rminor * jnp.cos(theta + self.triangularity * jnp.sin(theta)) * s,
            self.rminor * self.elongation * jnp.sin(theta) * s,
            phi
        ], axis=-1)
    
    def cartesian_position(self, s, theta, phi):
        return jsb.flux_surfaces.flux_surfaces_base._cylindrical_to_cartesian(self.cylindrical_position(s, theta, phi))

    def normal(self, s, theta, phi):
        raise NotImplementedError()
    
    def principal_curvatures(self, s, theta, phi):
        raise NotImplementedError()

We make a very low-res version to show the concept clearly:

In [ ]:
tokamak      = MillerTokamak(1.0, 1.0, 0.5, 0.2)
mesh_tokamak = jsb.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(tokamak, jnp.linspace(0.0,1.0,4), 0.0, 7)
# Convert it to a 2D mesh!
mesh_tokamak = (jnp.stack([mesh_tokamak[0][:,0], mesh_tokamak[0][:,2]], axis=-1), mesh_tokamak[1])
cross_sections_tokamak = CrossSectionsIndexed(indices = jnp.full((mesh_tokamak[1].shape[0],), 0), cross_section_data = cross_section)
domain_tokamak = Domain.from_mesh_and_cross_sections(mesh_tokamak, BasixLagrangianSimplex(1, 2).face_template, cross_sections_tokamak)
plot_domain(domain_tokamak, width=900, height=800)


Now let's pick an angle and show the corresponding graph:

In [ ]:
angle = jnp.array([1/jnp.sqrt(2), 1/jnp.sqrt(2), 0.0])

plot_domain(domain_tokamak, color_by_material = False, show_graph = True, graph_angle = angle, show_levels=True, mesh_opacity = 0.3, width=900, height=800)

which is no longer trivial. Let us solve and show the transport operator solution for an angle with both 1 and 2 degree finite elements:

In [ ]:
def _solve_and_show_2d(deg : int ):
    solution_domain_tokamak = SolutionDomain.from_element_and_domain(BasixLagrangianSimplex(deg,2), domain_tokamak)
    transport_operator      = create_transport_operator(solution_domain_tokamak, angle, VmapForward(), VmapSweep(), energy_group = 0)
    mass_operator           = MassOperator(MassData(solution_domain_tokamak.spatial_data.fem_matrices, 
                                                    solution_domain_tokamak.spatial_data.precomputed_geometric_data), 
                                            strategy = LaxMass(), additional_shape = ())
    psi_2d = transport_operator.inv(mass_operator.mv(jnp.ones_like(solution_domain_tokamak.elements_basis_shape)))

    from jax_sn.tools.plotly import plot_2d_solution
    return plot_2d_solution(
        solution_domain_tokamak,
        psi_2d,
        width=900, height=800, elevate = True, show_domain_mesh = True, show_element_boundaries = True, angle =angle, n_subdivisions = 1 if deg == 1 else 6
    )

_solve_and_show_2d(1)

In [ ]:
_solve_and_show_2d(2)